### Crear la Tabla results_movie en la capa "gold"

In [0]:
# %sql
# USE movie_silver

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
spark.sql("""
            CREATE TABLE IF NOT EXISTS movie_gold.results_movie
            (
               year_release_date INT, 
               country_name STRING, 
               company_name STRING, 
               budget FLOAT, 
               revenue FLOAT, 
               movie_id INT, 
               country_id INT, 
               company_id INT, 
               created_date DATE, 
               updated_date DATE 
            )
            USING DELTA       
          """)

DataFrame[]

In [0]:
spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW v_results_movies
        AS
        SELECT M.year_release_date, C.country_name, PCO.company_name,
                M.budget, M.revenue, M.movie_id, C.country_id, PCO.company_id
        FROM movie_silver.movies M
        INNER JOIN movie_silver.productions_countries PC ON M.movie_id = PC.movie_id
        INNER JOIN movie_silver.countries C ON C.country_id = PC.country_id
        INNER JOIN movie_silver.movies_companies MC ON M.movie_id = MC.movie_id
        INNER JOIN movie_silver.productions_companies PCO ON MC.company_id = PCO.company_id
        WHERE M.file_date = '{v_file_date}'
        """)

DataFrame[]

In [0]:
spark.sql(f"""
        MERGE INTO movie_gold.results_movie tgt
        USING v_results_movies src
        ON (tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id)
        WHEN MATCHED THEN
        UPDATE SET
            tgt.year_release_date = src.year_release_date,
            tgt.country_name = src.country_name,
            tgt.company_name = src.company_name,
            tgt.budget = src.budget,
            tgt.revenue = src.revenue,
            tgt.updated_date = current_timestamp
        WHEN NOT MATCHED THEN 
            INSERT (year_release_date, country_name, company_name, budget, revenue, movie_id, country_id, company_id, created_date)
            VALUES (year_release_date, country_name, company_name, budget, revenue, movie_id, country_id, company_id, current_timestamp)
            """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from v_results_movies

year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id
2012,United States of America,Spy Global Media,500000.0,625000.0,117942,214,31296
2014,United States of America,Moving Picture Company (MPC),1.7E8,7.73328629E8,118340,214,20478
2014,United Kingdom,Moving Picture Company (MPC),1.7E8,7.73328629E8,118340,162,20478
2014,United States of America,Marvel Studios,1.7E8,7.73328629E8,118340,214,420
2014,United Kingdom,Marvel Studios,1.7E8,7.73328629E8,118340,162,420
2014,United States of America,Bulletproof Cupid,1.7E8,7.73328629E8,118340,214,54850
2014,United Kingdom,Bulletproof Cupid,1.7E8,7.73328629E8,118340,162,54850
2014,United States of America,Revolution Sun Studios,1.7E8,7.73328629E8,118340,214,76043
2014,United Kingdom,Revolution Sun Studios,1.7E8,7.73328629E8,118340,162,76043
1998,United States of America,Forensic Films,300000.0,40542.0,118452,214,2813


In [0]:
%sql
SELECT * 
FROM movie_gold.results_movie

year_release_date,country_name,company_name,budget,revenue,movie_id,country_id,company_id,created_date,updated_date
2012,United States of America,Spy Global Media,500000.0,625000.0,117942,214,31296,2026-09-09,2026-09-09
2014,United States of America,Moving Picture Company (MPC),1.7E8,7.7332864E8,118340,214,20478,2026-09-09,2026-09-09
2014,United Kingdom,Moving Picture Company (MPC),1.7E8,7.7332864E8,118340,162,20478,2026-09-09,2026-09-09
2014,United States of America,Marvel Studios,1.7E8,7.7332864E8,118340,214,420,2026-09-09,2026-09-09
2014,United Kingdom,Marvel Studios,1.7E8,7.7332864E8,118340,162,420,2026-09-09,2026-09-09
2014,United States of America,Bulletproof Cupid,1.7E8,7.7332864E8,118340,214,54850,2026-09-09,2026-09-09
2014,United Kingdom,Bulletproof Cupid,1.7E8,7.7332864E8,118340,162,54850,2026-09-09,2026-09-09
2014,United States of America,Revolution Sun Studios,1.7E8,7.7332864E8,118340,214,76043,2026-09-09,2026-09-09
2014,United Kingdom,Revolution Sun Studios,1.7E8,7.7332864E8,118340,162,76043,2026-09-09,2026-09-09
1998,United States of America,Forensic Films,300000.0,40542.0,118452,214,2813,2026-09-09,2026-09-09
